# The goals of this notebook:

The NOAA dataset is missing location information. Although it includes variables for state and county FIPS codes, these codes are often incorrect (at least, many/most of them don't match values I can find online - e.g. at https://transition.fcc.gov/oet/info/maps/census/fips/fips.txt)

The goal of this notebook is to add FIPS codes for each NOAA event, based on:
- The path of the event (given by begin/end latitude and longitude)
- Text in the event narrative that indicates a county or weather station
- the county names contained in the dataset

**Note:** There are roughly 50,000 entries in the NOAA data that don't appear to correspond to counties. These include maratime areas, apparent sub-regions of counties (e.g., specific mountains or beaches). I'm not really sure how to deal with these.

We will then also add variables to the NOAA data to reflect the severity of the event. This includes:
- The overall duration of the event
- The duration of the event per county???
- Predictors that are already in the NOAA data?
- Maybe some ERA5 data?


Old notes (probably delete these):
- Creating a new dataframe in which each row is a county's experience of a weather event
- Identifying the time when the weather event entered and left the county (assuming constant velocity of the weather event)
- Computing the duration of the event in each county
- Looking up the ERA5 weather data closest to the... mean time of the event?
- But we also need to keep things in context of the overall severity of the event. Maybe also compute total duration for each event, and find some way to measure total severity from ERA5 data?
- (note: this isn't the same as loading all the ERA5 data and merging that with outages... it's just augmenting NOAA instead)

First, load the NOAA data and then convert the state names to abbreviations

In [1]:
import pandas as pd

df_events = pd.read_csv("../Data/NOAA_StormEvents/StormEvents_2014_2024.csv")

In [ ]:
# We'd like some code that converts state names into abbreviations
# One way to do this is using the 'us' package. But there are some dependency issues that are preventing me from installing it

#from us import states

In [2]:
# We'll use a dictionary to do the conversion.
# (There is a Pandas 'us' package that could do this for us, but I'm having dependency issues, so a dictionary it is...)

us_state_to_abbrev = {
    "ALABAMA": "AL",
    "ALASKA": "AK",
    "ARIZONA": "AZ",
    "ARKANSAS": "AR",
    "CALIFORNIA": "CA",
    "COLORADO": "CO",
    "CONNECTICUT": "CT",
    "DELAWARE": "DE",
    "FLORIDA": "FL",
    "GEORGIA": "GA",
    "HAWAII": "HI",
    "IDAHO": "ID",
    "ILLINOIS": "IL",
    "INDIANA": "IN",
    "IOWA": "IA",
    "KANSAS": "KS",
    "KENTUCKY": "KY",
    "LOUISIANA": "LA",
    "MAINE": "ME",
    "MARYLAND": "MD",
    "MASSACHUSETTS": "MA",
    "MICHIGAN": "MI",
    "MINNESOTA": "MN",
    "MISSISSIPPI": "MS",
    "MISSOURI": "MO",
    "MONTANA": "MT",
    "NEBRASKA": "NE",
    "NEVADA": "NV",
    "NEW HAMPSHIRE": "NH",
    "NEW JERSEY": "NJ",
    "NEW MEXICO": "NM",
    "NEW YORK": "NY",
    "NORTH CAROLINA": "NC",
    "NORTH DAKOTA": "ND",
    "OHIO": "OH",
    "OKLAHOMA": "OK",
    "OREGON": "OR",
    "PENNSYLVANIA": "PA",
    "RHODE ISLAND": "RI",
    "SOUTH CAROLINA": "SC",
    "SOUTH DAKOTA": "SD",
    "TENNESSEE": "TN",
    "TEXAS": "TX",
    "UTAH": "UT",
    "VERMONT": "VT",
    "VIRGINIA": "VA",
    "WASHINGTON": "WA",
    "WEST VIRGINIA": "WV",
    "WISCONSIN": "WI",
    "WYOMING": "WY",
    "DISTRICT OF COLUMBIA": "DC",
    "AMERICAN SAMOA": "AS",
    "GUAM": "GU",
    "NORTHERN MARIANA ISLANDS": "MP",
    "PUERTO RICO": "PR",
    "UNITED STATES MINOR OUTLYING ISLANDS": "UM",
    "U.S": "US"
}

def convert_state_name_to_abbrev(state_name):
    #Note that the state name needs to be in all capital letters
    return us_state_to_abbrev.get(state_name, "Unknown")

# Apply the function to convert the STATE variable in df_events to abbreviations
df_events['STATE_ABBREV'] = df_events['STATE'].apply(convert_state_name_to_abbrev)

Next, load the US Counties shapefile from the US Census.

This is available from https://www.census.gov/geographies/mapping-files/time-series/geo/cartographic-boundary.html

In [4]:
import geopandas

#Load the US Census Counties shapefile
counties = geopandas.read_file('../Data/cb_2023_us_county_500k')

#Concatenate STATEFP and COUNTFP and then convert to an integer
counties['FIPS'] = (counties['STATEFP'].astype(str) + counties['COUNTYFP'].astype(str)).astype(int)

Define a function that identifies all the FIPS in the path of the weather event.

This takes about 2 minutes to run.

We're going to assume that the path of the event is essentially linear. This is probably inaccurate, but we don't have any additional data (without doing something really clever with the ERA5 data) as an alternative

In [6]:
# Given a beginning point (given by BEGIN_LAT and BEGIN_LON) and an ending point (given by END_LAT and END_LON) from df_events, 
# identify which FIPS values from counties lie in the path between the beginning and ending points

def get_fips_from_path(row):
    #Get the beginning and ending values of longitude and latitude
    begin = (row['BEGIN_LON'], row['BEGIN_LAT'])
    end = (row['END_LON'], row['END_LAT'])

    #Convert them into geopandas points
    points = geopandas.points_from_xy([begin[0], end[0]], [begin[1], end[1]])

    #Get the path between the two points
    path = geopandas.GeoSeries(points)
    
    #Get the FIPS values from the counties shapefile that intersect with the path
    fips = counties[counties.geometry.intersects(path.union_all())]['FIPS'].tolist()
    
    return fips

#Apply get_fips_from_path to each row of df_events and create a new variable that lists the fips values
df_events['FIPS_from_Path'] = df_events.apply(get_fips_from_path, axis=1)

There are many rows in df_events that don't have beginning/end longitude/latitude values. However, the event_narrative variable sometimes mentions the name of a county.

The code below searches for county names in the event_narrative variable and matches them to a FIPS from the counties dataframe

Note that this takes almost 5 minutes to run

In [7]:
def get_fips_from_narrative(row):
    #Get the state from the row
    state = row['STATE_ABBREV']
    
    #Get the event narrative from the row
    narrative = row['EVENT_NARRATIVE']

    #Create a new dataframe from county_fips that has the same state
    counties_state = counties[counties['STUSPS'] == state]
    
    #Identify any word in the EVENT_NARRATIVE variable that matches the COUNTYNAME variable in county_fips for the same STATE
    fips = []
    for index, row in counties_state.iterrows():
        #if narrative is not NaN:
        if pd.notna(row['NAME']) and pd.notna(narrative):
            if row['NAME'] in narrative:
                fips.append(row['FIPS'])    
    return fips

#Apply the function to each row of df_events that lack latitude data to add a FIPS code
df_events['FIPS_from_Countyname_Narrative'] = df_events[df_events['BEGIN_LAT'].isna()].apply(get_fips_from_narrative, axis=1)

In addition to county names, sometimes the event_narrative variable refers to (what seem to be) various sorts of weather stations.

Like the county names, we can try to use these names to identify the FIPS where the weather event was reported.

Use the list of weather stations from Meteostat (https://github.com/meteostat/weather-stations?tab=readme-ov-file) to further identify locations

Note that lists of additional weather stations are available from NOAA: https://www.ncei.noaa.gov/access/homr/#. However, I've had trouble reliably identifying examples of weather stations from the downloaded lists, so sticking with meteostat for now.

In the code below, we'll load the Meteostat list, do some cleaning, add a geometry variable so we can merge it with the counties dataframe, and look up the FIPS code for each station from the counties data

In [9]:
#Load the Meteostat list of weather stations
import pandas as pd
df_meteostat_list = pd.read_json('../Data/full.json')

#Restrict the country variable to "US"
df_meteostat_list = df_meteostat_list[df_meteostat_list['country'] == "US"]

#Convert the name variable into strings and strip the text {'en': '
df_meteostat_list['name'] = df_meteostat_list['name'].astype(str).str.strip("{'en': '").str.strip("'").str.strip("'}")

#Extract the icao value from the identifiers variable
df_meteostat_list['icao'] = df_meteostat_list['identifiers'].apply(lambda x: x['icao'])

#Extract the latitude, longitude, and elevation from the location variable
df_meteostat_list['latitude'] = df_meteostat_list['location'].apply(lambda x: x['latitude'])
df_meteostat_list['longitude'] = df_meteostat_list['location'].apply(lambda x: x['longitude'])
df_meteostat_list['elevation'] = df_meteostat_list['location'].apply(lambda x: x['elevation'])

#Drop the identifiers, location, and inventory variables
df_meteostat_list = df_meteostat_list.drop(columns=['country','identifiers', 'location', 'inventory'])

#Convert the id, name, region, and icao variables into strings
df_meteostat_list['id'] = df_meteostat_list['id'].astype(str)
df_meteostat_list['name'] = df_meteostat_list['name'].astype(str)
df_meteostat_list['region'] = df_meteostat_list['region'].astype(str)
df_meteostat_list['icao'] = df_meteostat_list['icao'].astype(str)

#Convert df_meteostat_list into a geopandas dataframe
df_meteostat_list = geopandas.GeoDataFrame(df_meteostat_list, geometry=geopandas.points_from_xy(df_meteostat_list['longitude'], df_meteostat_list['latitude']))

#Set the CRS as EPSG 4269
# Note: I'm not entirely sure this is the CRS for the data - I couldn't find any info directly from the meteostat website
df_meteostat_list.set_crs(epsg=4269, inplace=True)

#Perform a spatial join between the counties and the df_meteostat_list dataframes to add the FIPS variable to df_meteostat_list
df_meteostat_list['FIPS'] = geopandas.sjoin(df_meteostat_list, counties, how='left', predicate='intersects')['FIPS']

Next we'll define a function that will look through the event narrative and look for mentions of weather stations, then find the corresponding FIPS code

Note that the next code chunk takes nearly 4 minutes to run.

In [10]:
def get_fips_from_meteostat(df_events_row):
    #Get the state from the row
    state = df_events_row['STATE_ABBREV']
    
    #Get the event narrative from the row
    narrative = df_events_row['EVENT_NARRATIVE']
    
    #Create a new dataframe from df_meteostat_list that has the same state
    df_meteostat_list_state = df_meteostat_list[df_meteostat_list['region'] == state]
    
    #Identify any word in the EVENT_NARRATIVE variable that matches the id variable, the icao variable, or the name variable in df_meteostat_list for the same STATE
    fips = []
    for index, row in df_meteostat_list_state.iterrows():
        if pd.notna(narrative):
            if str(row['name']) in narrative or str(row['icao']) in narrative or str(row['id']) in narrative:
                fips.append(row['FIPS'])
    return fips

#Apply the function to each row of df_events that lack latitude data and create a new variable that lists the fips values
df_events['FIPS_from_Meteostat'] = df_events[df_events['BEGIN_LAT'].isna()].apply(get_fips_from_meteostat, axis=1)

The NOAA data includes columns for STATE_FIPS and CZ_FIPS. The stat FIPS values appear correct, but the county FIPS values don't appear to match what I can find online.

For consistency, we can look up the FIPS for the county using the STATE_ABBREV and CZ_NAME variables

Note that this code takes 4 1/2 minutes to run

In [11]:
def get_fips_from_czname(row):
    #Get the state from the row
    state = row['STATE_ABBREV']
    
    #Get the county name from the row
    czname = row['CZ_NAME']
    
    #Create a new dataframe from counties that has the same state
    counties_state = counties[counties['STUSPS'] == state]
    
    fips = []
    for index, row in counties_state.iterrows():
        #if COUNTYNAME is not NaN and narrative is not NaN:
        if pd.notna(czname):
            #Convert row['NAME'] to all capitals
            uppername = row['NAME'].upper()
            if uppername in czname:
                fips.append(row['FIPS'])
    #print(fips)
    
    return fips

#Apply get_fips_from_narrative to each row of df_events and create a new variable that lists the fips values
# But do this only for rows of df_events that lack latitude and/or longitude data and only for rows of df_events that have a value for STATE_ABBREV
df_events['FIPS_from_CZNAME'] = df_events[df_events['BEGIN_LAT'].isna()].apply(get_fips_from_czname, axis=1)

Now, we'll create a list of all the FIPS associated with each event

In [15]:
#In the df_events data frame, create a new list of values by combining the values of the variables FIPS_from_Path, FIPS_from_Countyname_Narrative, FIPS_from_Meteostat, and FIPS_from_CZNAME, dropping duplicate values
df_events['FIPS'] = df_events[['FIPS_from_Path', 'FIPS_from_Countyname_Narrative', 'FIPS_from_Meteostat', 'FIPS_from_CZNAME']].apply(lambda x: list(set(x.dropna().sum())), axis=1)

In [18]:
#Make a dataframe from df_events where FIPS is an empty list and save it as a csv
df_events[df_events['FIPS'].apply(lambda x: len(x) == 0)].to_csv('../Data/df_events_missing_fips.csv', index=False)

In [1]:
import fsspec

fs = fsspec.filesystem('gs')
fs.ls('gs://weatherbench2/datasets/era5/')

['weatherbench2/datasets/era5/',
 'weatherbench2/datasets/era5/1959-2022-1h-240x121_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-1h-360x181_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-128x64_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-128x64_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-1440x721.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-240x121_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-512x256_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x32_equiangular_with_poles_conservative.zarr',
 'weatherbench2/datasets/era5/1959-2022-6h-64x33.zarr',
 'weatherbench2/datasets/era5/1959-2022-full_37-1h-0p25deg-chunk-1.zarr-v2',
 'weatherbench2/datasets/era5/1959-2022-full_37-6h-0p25deg-chu

Next, we'll load the 6-hour downsampled data set

In [41]:
import xarray as xr

reanalysis = xr.open_zarr(
    'gs://weatherbench2/datasets/era5/1959-2023_01_10-wb13-6h-1440x721_with_derived_variables.zarr', 
    chunks={'time': 48},
    consolidated=True,
    decode_timedelta=True    
)

Let's reduce the size by only including data since 1/1/2014 and only our target variables:
- 10m_wind_speed
- 2m_temperature
- snow_depth
- total_precipitation_12hr
- total_precipitation_24hr
- total_precipitation_6hr
- wind_speed

In [ ]:
#Select time values since 1/1/2014 and the following variables: 
features = [
    'latitude', 
    'longitude', 
    'time', 
    '10m_wind_speed', 
    '2m_temperature', 
    'snow_depth', 
    'total_precipitation_12hr', 
    'total_precipitation_24hr', 
    'total_precipitation_6hr', 
]

reanalysis = reanalysis.sel(time=slice('2014', '2021'))[features]

Next, we'll restrict our data to county centroids (in the counties_centroids.csv file created by the convert_NWS_shapefile_to_county_centroids notebook).

In the ERA5 coordinate system, latitude values are "normal" but longitude values are expressed as values within [0, 360] with respect to the Greenwich Prime Meridian (i.e., instead of [-180, 180]). Since our county centroid data are all West of the Prime Meridian, we can simply adjust their longitude values with an auxiliary function.

In [43]:
#Use xarray to load the file ../Data/counties_centroids.csv
import pandas as pd
counties_centroids = pd.read_csv('../Data/counties_centroids.csv')

counties_centroids['LON'] = counties_centroids['LON'].astype(float)
counties_centroids['LAT'] = counties_centroids['LAT'].astype(float)

#The function below converts "standard" longitude values to ERA5 longitude values
def lon_to_360(dlon: float) -> float:
  return ((360 + (dlon % 360)) % 360)

counties_centroids['LON'] = counties_centroids['LON'].apply(lon_to_360)

In [14]:
#Create a new variable 'position' in counties_centroids that is the ordered pair from LON and LAT
counties_centroids['position'] = list(zip(counties_centroids['LON'], counties_centroids['LAT']))

In [44]:
#Create a new DataArray from counties_centroids with the LON and LAT values as longitude and latitude coordinates and FIPS as the data
counties_centroids_da = xr.DataArray(
    counties_centroids['FIPS'].values,
    coords={
        'longitude': ('points', counties_centroids['LON'].values),
        'latitude': ('points', counties_centroids['LAT'].values)
    },
    dims='points'
)

In [45]:
#Restrict the reanalysis data set to coordinates in counties_centroids_da
reanalysis_counties = reanalysis.sel(
    longitude=reanalysis.longitude.isin(counties_centroids_da.longitude),
    latitude=reanalysis.latitude.isin(counties_centroids_da.latitude)
)

We can try saving the zarr file locally as a NetCDF file. The size of the data is roughly 9 GB. However, I've let it run for several hours without it completing.

I've run into roadblocks trying to save locally as a zarr file; I keep getting a "TypeError(f"Expected a BytesBytesCodec. Got {type(data)} instead.")" error. From what I can tell, this appears to be related to zarr 3.

In [14]:
# Compute the size of the reanalysis_counties dataset in GB
#print(f'size: {reanalysis_counties.nbytes / (1024 ** 3)} GiB')

#Export reanalysis_counties to NetCDF
#reanalysis_counties.to_netcdf('../Data/reanalysis_counties.nc')

In [12]:
#Load the ../Data/eaglei_data/eaglei_outages_with_county_info.parquet data set
import pyarrow.parquet as pq
eaglei_outages = pq.read_table('../Data/eaglei_data/eaglei_outages_with_county_info.parquet').to_pandas()

In [15]:
#In eaglei_outages convert centroid to the coordinate pair LON, LAT
eaglei_outages['LON'] = eaglei_outages['centroid'].apply(lambda x: x[0])
eaglei_outages['LAT'] = eaglei_outages['centroid'].apply(lambda x: x[1])